In [1]:
import torch 
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer 
from peft import get_peft_model, LoraConfig
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/home/omar-elmasaoudi/miniconda3/envs/ml_dev_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


The CorDA tuning step looked at how to tune a CorDA model efficiently using hugging face's high level tuning library and training loops. In this notebook, I will demonstrate what happens under the hood when tuning a smaller model -- for testing purposes -- and as well as the effectiveness of adapter tuning on downstream tasks. 

### Load the gpt-neo-125M model 

In [2]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125m")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")
print(model)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [21]:
GPT_NEO_BLOCK = model.transformer.h
attention_modules = []
for gpt_block in GPT_NEO_BLOCK:
    attention_modules.append(gpt_block.attn.attention)

#Example of the models attention head 
print(attention_modules[0])

GPTNeoSelfAttention(
  (attn_dropout): Dropout(p=0.0, inplace=False)
  (resid_dropout): Dropout(p=0.0, inplace=False)
  (k_proj): Linear(in_features=768, out_features=768, bias=False)
  (v_proj): Linear(in_features=768, out_features=768, bias=False)
  (q_proj): Linear(in_features=768, out_features=768, bias=False)
  (out_proj): Linear(in_features=768, out_features=768, bias=True)
)


In [22]:
#We'll use an example attention head to demonstrate the Linear layers we're performing Lora on 
attention_1 = attention_modules[0]
k_proj = attention_1.k_proj
## Print the raw tensor of the K proj matrix and confirm it's shape 
print(k_proj.weight.shape)
print(k_proj.weight.dtype)

torch.Size([768, 768])
torch.float32


### Save model's old params locally

We'll save the params of each block's q, k, v projections 

In [7]:
import torch as pt
import os

EXPERIMENT_NUMBER = 0
proj_modules = ["k_proj", "q_proj", "v_proj"]

for idx, attention in enumerate(attention_modules):
    block_dir = f"../model/lora/base_model_attention_proj_weights/block_{idx}"
    os.makedirs(block_dir, exist_ok=True)
    pt.save(attention.k_proj.weight.detach().cpu(), f"{block_dir}/k_proj.pt")
    pt.save(attention.q_proj.weight.detach().cpu(), f"{block_dir}/q_proj.pt")
    pt.save(attention.v_proj.weight.detach().cpu(), f"{block_dir}/v_proj.pt")


### Define LoRa Initializer

In [ ]:

class LoRaHelper(nn.Module):
    def __init__(self,og_weights, alpha:int , in_dims: int = 768, out_dims: int = 768, r:int = 4 ):
        super().__init__()
        self.W = og_weights
        self.alpha = alpha 
        self.r = r 
        #B has dims R^{d * r} init all of B to 0's  
        self.B = torch.zeros(out_dims, r)
        self.init_A(r = r , k = in_dims)
        #.Parameter subclasses the tensor class and tell us that these tensor are learnable params inside of the nn.Module
        self.A = nn.Parameter(self.A)
        self.B = nn.Parameter(self.B)
         
    def forward(self, x):
        delta_w = self.B @ self.A
        F = self.W(x)
        G = torch.matmul(delta_w, x) * (self.alpha / self.r)
        h = F + G  
        return h
        
    # A has dims R ^ {r * k }
    def init_A(self, r: int, k:int ):
        self.A = torch.empty(r, k)
        self.A = torch.nn.init.normal(self.A, mean=0.0, std=0.02)
        

### Define Hyper params 

In [8]:
EPOCHS = 5 
alpha = 0.02
lr = 0.01
batch_size = 2 

### Load the dataaset 

In [9]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
ds = ds.with_format("torch")
dataloader = DataLoader(ds,batch_size=batch_size, shuffle=True)

### Freeze the models parameters

In [ ]:
for param in model.parameters():
    param.requires_grad = False

### Quick Sanity Check for param freezing 

In [ ]:
for i in model.parameters():
    if i.requires_grad == True:
        print("There is an error in grad freezing")

### Unfreeze k,q, and v projection layers

In [36]:
# For each attention head inside of the GPTNeoAttention Module unfreeze the gradients
for attention_head in attention_modules:
    for k_proj_params in attention_head.k_proj.parameters():
        k_proj_params.requires_grad = True 
    for q_proj_params in attention_head.q_proj.parameters():
        q_proj_params.requires_grad = True 
    for v_proj_params in attention_head.v_proj.parameters():
        v_proj_params.requires_grad = True 
        

In [43]:
for name, p in model.named_parameters():
    if p.requires_grad == True:
        print(f"This is the module that has unfrozen params: {name}")

This is the module that has unfrozen params: transformer.h.0.attn.attention.k_proj.weight
This is the module that has unfrozen params: transformer.h.0.attn.attention.v_proj.weight
This is the module that has unfrozen params: transformer.h.0.attn.attention.q_proj.weight
This is the module that has unfrozen params: transformer.h.1.attn.attention.k_proj.weight
This is the module that has unfrozen params: transformer.h.1.attn.attention.v_proj.weight
This is the module that has unfrozen params: transformer.h.1.attn.attention.q_proj.weight
This is the module that has unfrozen params: transformer.h.2.attn.attention.k_proj.weight
This is the module that has unfrozen params: transformer.h.2.attn.attention.v_proj.weight
This is the module that has unfrozen params: transformer.h.2.attn.attention.q_proj.weight
This is the module that has unfrozen params: transformer.h.3.attn.attention.k_proj.weight
This is the module that has unfrozen params: transformer.h.3.attn.attention.v_proj.weight
This is th

### Create tuning loop

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

loss_fn = torch.nn.CrossEntropyLoss()
lora_module = LoRaHelper(model.parameters())
for epoch in range(EPOCHS):    
    model.train()
    for step, batch in enumerate(tqdm(dataloader)):
        X = batch["text"]
        X = [t for t in X if t.strip() != ""]
        if len(X) == 0:
            continue
        tokenized_batch = tokenizer(X, padding=True,truncation=True, return_tensors="pt", max_length=96).to(model.device)
        input_ids = tokenized_batch["input_ids"].to(model.device)
        attention_mask = tokenized_batch["attention_mask"].to(model.device)
        
        # This is the forward pass
        preds = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        logits = preds["logits"]
        loss = preds.loss
        
        # Here set the optimizer to no grad updates
        optimizer.zero_grad()
        #Run back prop once for the step 
        loss.backward()
        #Use the defined optimizer to update the grads accordingly 
        optimizer.step()
        
        if step % 50 == 0:
            print(f"epoch {epoch} step {step} loss: {loss.item()}" )
        
    # model.eval()

  0%|          | 1/18359 [00:01<9:03:47,  1.78s/it]

epoch 0 step 0 loss: 5.495945930480957


  0%|          | 51/18359 [01:26<12:19:59,  2.43s/it]

epoch 0 step 50 loss: 1.7734684944152832


  0%|          | 62/18359 [01:35<7:49:56,  1.54s/it] 


KeyboardInterrupt: 